In [5]:
import pandas as pd

In [6]:
wdc = pd.read_json("data/wdc/wdcproducts80cc20rnd000un_train_small.json.gz", compression="gzip", lines=True)

In [12]:
wdc["label"].value_counts()

label
0    2000
1     500
Name: count, dtype: int64

In [4]:
left_cols = [c for c in wdc.columns if c.endswith("_left")]
right_cols = [c for c in wdc.columns if c.endswith("_right")]

df_left = wdc[left_cols].rename(columns=lambda c: c.removesuffix("_left"))
df_right = wdc[right_cols].rename(columns=lambda c: c.removesuffix("_right"))

df_left.to_csv("data/wdc/wdc_train_large_left.csv", index=False)
df_right.to_csv("data/wdc/wdc_train_large_right.csv", index=False)

print(f"Left:  {df_left.shape}")
print(f"Right: {df_right.shape}")
df_left.head()

Left:  (19835, 7)
Right: (19835, 7)


,id,brand,title,description,price,priceCurrency,cluster_id
0,533407,Western Digital,WD Blue SN550 1TB M.2 SSD,"(Solid, 1TB, Blue, Drive), M.2, SN550, SSD, St...",119,EUR,156996
1,22587878,None,TP-Link AC600 Nano Wireless USB Adapter,"High Speed WiFi, Dual Band Wireless, Nano desi...",12.48,GBP,373003
2,25725051,AMD,AMD Ryzen 5 2600X - 4.25 GHz - 6-core - 12 thr...,"IcecatLive.getDatasheet('#IcecatLive',{'UserNa...",296.39,EUR,9046
3,2952392,Corsair,Carbide 175R RGB Windowed Mid-Tower Case Tempe...,Corsair Carbide 175R RGB Windowed Mid-Tower Ca...,69.90,EUR,643961
4,91577206,None,Switch D-Link - Web smart - switch - 24 porte ...,24-PORTS POE 10/100/1000MBPS WITH 4 X 1000BASE...,4.0099E2,EUR,56526


In [3]:
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

In [ ]:


embed_cols = ["title", "brand", "description"]

def combine_text(df, cols):
    return df[cols].fillna("").astype(str).agg(" ".join, axis=1)

# Build JSONL batch requests for left and right
for side, df_side in [("left", df_left), ("right", df_right)]:
    texts = combine_text(df_side, embed_cols)
    jsonl_path = f"data/wdc/batch_embed_{side}.jsonl"

    with open(jsonl_path, "w") as f:
        for i, text in enumerate(texts):
            req = {
                "custom_id": f"{side}-{i}",
                "method": "POST",
                "url": "/v1/embeddings",
                "body": {
                    "model": "text-embedding-3-small",
                    "input": text,
                },
            }
            f.write(json.dumps(req) + "\n")

    # Upload and submit batch
    batch_file = client.files.create(file=open(jsonl_path, "rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/embeddings",
        completion_window="24h",
    )
    print(f"{side}: batch {batch.id}  status={batch.status}")

left: batch batch_69949a5a3adc8190a5587ec6b95c6878  status=validating
right: batch batch_69949a631c2c8190b01caa9ca2f49d11  status=validating


In [4]:
# --- Run this cell after the batches complete ---
# Replace with your actual batch IDs printed above
BATCH_IDS = {
    "left": "batch_69949a5a3adc8190a5587ec6b95c6878",
    "right": "batch_69949a631c2c8190b01caa9ca2f49d11",
}

import numpy as np

for side, batch_id in BATCH_IDS.items():
    batch = client.batches.retrieve(batch_id)
    print(f"{side}: status={batch.status}")

    if batch.status != "completed":
        print(f"  Not ready yet — re-run this cell later.")
        continue

    # Download results
    content = client.files.content(batch.output_file_id).text
    results = [json.loads(line) for line in content.strip().split("\n")]

    # Sort by custom_id index to preserve row order
    results.sort(key=lambda r: int(r["custom_id"].split("-")[1]))

    embeddings = np.array(
        [r["response"]["body"]["data"][0]["embedding"] for r in results],
        dtype=np.float32,
    )
    np.save(f"data/wdc/wdc_{side}_embeddings.npy", embeddings)
    print(f"  Saved {embeddings.shape} to data/wdc/wdc_{side}_embeddings.npy")

left: status=completed
  Saved (19835, 1536) to data/wdc/wdc_left_embeddings.npy
right: status=completed
  Saved (19835, 1536) to data/wdc/wdc_right_embeddings.npy


In [1]:
# --- Load GS (gold standard) test set and submit GPT-5.2 labeling batch ---
gs = pd.read_json(
    "data/wdc/wdcproducts80cc20rnd100un_gs.json.gz", compression="gzip", lines=True
)
print(f"GS pairs: {gs.shape[0]}")
print(f"Label distribution:\n{gs['label'].value_counts()}")
gs.head(3)

NameError: name 'pd' is not defined

In [24]:
# Build batch JSONL — prompt tuned for balanced precision/recall
FIELDS = ["title", "brand", "description", "price", "priceCurrency"]
MODEL = "gpt-5.2"

SYSTEM_PROMPT = """You are an expert product matcher.
Task: decide whether two listings refer to the same underlying commercial product/model.

Return exactly one JSON object:
{"match": true|false}

Guidelines:
1) Use title/description/brand/model identifiers as primary evidence.
2) Treat language differences, abbreviations, formatting noise, and regional wording as equivalent.
3) When exact model code is missing on one side, allow a match if the remaining evidence strongly points to the same product line/model.
4) Price and currency are weak signals only.
5) Mark match=false only when there is clear contradictory evidence of different products, such as:
   - different model/part number that conflict
   - different core capacity/size/color/generation/spec that conflict
   - clearly different product type/category
6) Do NOT require perfect detail overlap. One listing may be shorter, generic, or partially specified.
7) If there is strong positive evidence and no hard contradiction, prefer match=true.

Output rules:
- Valid JSON only.
- Only the key "match".
- No extra text."""

def serialize_record(row, fields, suffix):
    data = {}
    for f in fields:
        col = f"{f}_{suffix}"
        val = row.get(col)
        if pd.notna(val):
            val_str = str(val)
            if len(val_str) > 350:
                val_str = val_str[:350] + "..."
            data[f] = val_str
    return json.dumps(data, ensure_ascii=False)

jsonl_path = "data/wdc/batch_llm_label_gs.jsonl"
with open(jsonl_path, "w") as f:
    for i, row in gs.iterrows():
        left_data = serialize_record(row, FIELDS, "left")
        right_data = serialize_record(row, FIELDS, "right")
        user_msg = (
            f"Listing A: {left_data}\n"
            f"Listing B: {right_data}\n"
            "Do these refer to the same underlying product model?"
        )
        req = {
            "custom_id": f"gs-{i}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                "temperature": 0,
            },
        }
        f.write(json.dumps(req) + "\n")

print(f"Wrote {len(gs)} requests to {jsonl_path}")

# Upload and submit batch
batch_file = client.files.create(file=open(jsonl_path, "rb"), purpose="batch")
batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)
print(f"Batch ID: {batch.id}  status={batch.status}")



Wrote 4500 requests to data/wdc/batch_llm_label_gs.jsonl
Batch ID: batch_6994e76a3ad88190a09023fb490c0361  status=validating


In [26]:
# --- Run after batch completes: retrieve results and compare to gold labels ---
LLM_BATCH_ID = "batch_6994e76a3ad88190a09023fb490c0361"  # replace with batch.id from above

batch_status = client.batches.retrieve(LLM_BATCH_ID)
print(f"Status: {batch_status.status}")

if batch_status.status != "completed":
    print("Not ready yet — re-run this cell later.")
else:
    content = client.files.content(batch_status.output_file_id).text
    results = [json.loads(line) for line in content.strip().split("\n")]
    results.sort(key=lambda r: int(r["custom_id"].split("-")[1]))

    # Parse LLM predictions
    preds = []
    for r in results:
        idx = int(r["custom_id"].split("-")[1])
        text = r["response"]["body"]["choices"][0]["message"]["content"]
        try:
            match = json.loads(text).get("match", False)
        except json.JSONDecodeError:
            match = False
        preds.append({"idx": idx, "pred": int(bool(match))})

    preds_df = pd.DataFrame(preds).set_index("idx").sort_index()

    # Align with gold labels
    gs_aligned = gs.loc[preds_df.index]
    preds_df["gold"] = gs_aligned["label"].values
    preds_df["pair_id"] = gs_aligned["pair_id"].values

    # Metrics
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

    acc = accuracy_score(preds_df["gold"], preds_df["pred"])
    prec = precision_score(preds_df["gold"], preds_df["pred"])
    rec = recall_score(preds_df["gold"], preds_df["pred"])
    f1 = f1_score(preds_df["gold"], preds_df["pred"])

    print(f"\nGPT-5.2 on WDC GS (100% corner cases):")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  Pairs:     {len(preds_df)}")

    # Save predictions
    preds_df.to_csv("data/wdc/gpt52_gs_predictions.csv")
    print("\nSaved to data/wdc/gpt52_gs_predictions.csv")

Status: completed

GPT-5.2 on WDC GS (100% corner cases):
  Accuracy:  0.9460
  Precision: 0.9673
  Recall:    0.5320
  F1:        0.6865
  Pairs:     4500

Saved to data/wdc/gpt52_gs_predictions.csv
